# Demo 11 — Fitting double-lined radial velocities

Fit the bundled RV observations with least squares, run a short MCMC demonstration, then save and reload results without losing the distinction between ensemble steps and flat sample rows. Demo 10 explains data preparation; Demo 12 combines RV information with photometry.

The RV fitting task uses a **kinematic centre-of-mass model**. It needs no atmosphere or limb-darkening tables. Radiometric RVs, spots and eclipse-related line-profile effects discussed in Demo 9 are not part of this fitting model.

In [ ]:
import json
from pathlib import Path

import astropy.units as u
import numpy as np
import pandas as pd

from elisa import settings
from elisa.analytics import RVBinaryAnalyticsTask, RVData
from elisa.conf.enums import DiscardUnit

tutorial_dir = Path.cwd()
if not (tutorial_dir / "demo_data").is_dir():
    tutorial_dir = tutorial_dir / "jupyter_tutorials"
data_dir = tutorial_dir / "demo_data"
output_dir = Path("demo11-output").resolve()
output_dir.mkdir(exist_ok=True)


## Observations and parameterization

The primary and secondary datasets can have different observation times. Their values and errors are supplied in km/s and converted together to internal m/s. The bundled timestamps and the epoch below share an offset day convention; ELISA does not infer time standards or apply barycentric corrections. Preserve the measured velocity zero point when fitting `gamma`.

In [ ]:
rv_data = {}
for component, filename in {"primary": "rv1.dat", "secondary": "rv2.dat"}.items():
    rv_data[component] = RVData.load_from_file(
        filename=str(data_dir / "rv_data" / filename),
        name=f"{component} RV",
        x_unit=u.d,
        y_unit=u.km / u.s,
        data_columns=(0, 1, 2),
    )
print(pd.DataFrame([
    {
        "component": component,
        "observations": data.x_data.size,
        "median_sigma_km_per_s": np.median(data.y_err) / 1000.0,
    }
    for component, data in rv_data.items()
]).to_string(index=False))


This example uses the `community` parameterization: `asini` is $(a_1+a_2)\sin i$, the **relative binary orbit's** projected semi-major axis; `mass_ratio` is $q=M_2/M_1$. RVs alone constrain projected dimensions and masses, not inclination or the individual true masses. The alternative `standard` combination supplies component masses and inclination.

Free parameters have `fixed=False` and finite `min`/`max` bounds. Values and bounds use the declared unit. Period and epoch are fixed here using external information. Close to a circular orbit, the argument of periastron is poorly determined; its apparent uncertainty should not be interpreted independently of eccentricity.

In [ ]:
rv_init_params = {
    "system":{
        "eccentricity": {
            "value": 0.03,
            "fixed": False,
            "min": 0.0,
            "max": 0.1,
        },
        "asini": {
            "value": 12,
            "fixed": False,
            "min": 8.0,
            "max": 15,
            "unit": u.solRad,
        },
        "mass_ratio": {
            "value": 1.0,
            "fixed": False,
            "min": 0.9,
            "max": 1.2,
        },
        "argument_of_periastron": {
            "value": 170,
            "fixed": False,
            "min": 0,
            "max": 360,
            "unit": u.deg,
        },
        "gamma": {
            "value": -25.0,
            "fixed": False,
            "min": -50.0,
            "max": -0.0,
            "unit": u.km/u.s,
        },
        "period": {
            "value":2.47028,
            "fixed": True,
        },
        "primary_minimum_time": {
            "value": 54953.900507000006,
            "fixed": True,
            "unit": u.d,
        },
    },
    "nuisance": {
        "ln_f": {
            "value": -5,
            "fixed": False,
            "min": -10,
            "max": 0,
            "unit": None,
        },
    },
}

## Least squares: a starting solution

The solver minimizes the individual standardized residuals $(y_i-m_i)/\sigma_i$. For the default linear loss its cost is half their sum of squares. `ln_f` belongs to the MCMC likelihood and is not fitted by this least-squares objective. Missing or miscalibrated measurement errors change the statistical interpretation.

The parameter result contains fitted values, units and bounds. It is suitable for initializing MCMC but does not provide posterior intervals. Recognized nonphysical trial models receive a penalty; invalid setup and unexpected errors propagate.

In [ ]:
least_squares_task = RVBinaryAnalyticsTask(
    data=rv_data,
    method="least_squares",
)
print(least_squares_task.FIT_PARAMS_COMBINATIONS)

least_squares_result = least_squares_task.fit(
    x0=rv_init_params,
    max_nfev=100,
)
least_squares_task.plot.model(
    start_phase=-0.5,
    stop_phase=0.5,
    y_axis_unit=u.km / u.s,
)
least_squares_path = output_dir / "rv_least_squares.json"
least_squares_task.save_result(str(least_squares_path))


Inspect the model and residual panels for systematic departures, component-specific offsets and outliers. A high $R^2$ alone does not validate the physical or noise model. Trying several initial solutions is useful when eccentricity and periastron orientation are correlated.

The projected semi-major axis, period, eccentricity and mass ratio also determine the component RV semi-amplitudes. Here they are calculated from the fitted values without a second fit:

In [ ]:
system_result = least_squares_result["system"]
asini_m = (system_result["asini"]["value"] * u.solRad).to_value(u.m)
period_s = (system_result["period"]["value"] * u.d).to_value(u.s)
eccentricity = system_result["eccentricity"]["value"]
mass_ratio = system_result["mass_ratio"]["value"]
k_sum = 2.0 * np.pi * asini_m / (period_s * np.sqrt(1.0 - eccentricity**2))
print(f"K1 = {k_sum * mass_ratio / (1.0 + mass_ratio) / 1000.0:.3f} km/s")
print(f"K2 = {k_sum / (1.0 + mass_ratio) / 1000.0:.3f} km/s")


## MCMC: likelihood, priors and initialization

For supplied uncertainties, the Gaussian likelihood uses $s_i^2=\sigma_i^2[1+\exp(2\,\mathrm{ln\_f})]$, including its normalization term. `ln_f` is a logarithmic **fractional inflation of measurement errors**, not an additive velocity jitter or a model-flux term. Its bounds here restrict the possible inflation. Without a parameter `sigma`, the prior is uniform inside its bounds. A supplied `sigma` instead gives a bounded Gaussian prior centered on the initial value; re-centering such a prior on a data-derived least-squares solution requires scientific justification.

By default the first walker starts at the normalized initial vector and the others are drawn uniformly across normalized bounds. Passing the least-squares result therefore does not concentrate the entire ensemble near that solution. A custom `initial_state` must have shape `(nwalkers, ndim)` and contain normalized coordinates in `[0, 1]` in the fitter's parameter order.

The following run uses 24 walkers, 200 burn-in steps and 1,000 **production** steps. This demonstrates the workflow; the step count alone does not establish convergence. Use temporal diagnostics and independent starts to assess whether longer sampling is needed.

In [ ]:
mcmc_task = RVBinaryAnalyticsTask(
    data=rv_data,
    method="mcmc",
)
chain_path = output_dir / "rv_production_chain.json"
raw_result_path = output_dir / "rv_mcmc_all_production.json"
nwalkers = 24
production_steps = 1000
burn_in_steps = 200

# Limit notebook sampling to one process; settings are restored on exit.
with settings.override(
    NUMBER_OF_MCMC_PROCESSES=1,
    LOG_CONFIG="fit",
):
    mcmc_result = mcmc_task.fit(
        x0=least_squares_result,
        nwalkers=nwalkers,
        nsteps=production_steps,
        burn_in=burn_in_steps,
        save=True,
        fit_id=str(chain_path),
        progress=False,
    )
mcmc_task.save_result(str(raw_result_path))


## Burn-in is not saved production

`burn_in` runs before production with storage disabled, and the sampler is reset before the `nsteps` production iterations. Those burn-in steps are already absent from the saved chain. If omitted, `burn_in` defaults to `int(nsteps / 10)`.

The current chain export stores normalized flat samples plus `chain_shape` and `chain_order` metadata. The temporal shape is `(step, walker, parameter)`; flattening uses step-major order with walkers varying fastest. For 1,000 steps and 24 walkers there are 24,000 flat rows, not 24,000 consecutive iterations of one walker. All raw sample values need their saved normalization bounds to recover physical parameter values.

In [ ]:
fit = mcmc_task.fit_cls
print("Temporal chain:", fit.chain.shape)
print("Flat samples:", fit.flat_chain.shape)
print("Columns:", fit.variable_labels)
np.testing.assert_equal(fit.chain.shape[:2], (production_steps, nwalkers))
np.testing.assert_array_equal(
    fit.flat_chain,
    fit.chain.reshape(-1, fit.chain.shape[-1]),
)
with chain_path.open() as stream:
    chain_metadata = json.load(stream)
print("Saved shape:", chain_metadata["chain_shape"])
print("Saved order:", chain_metadata["chain_order"])


## Diagnose before choosing an additional cut

Default traces use the temporal chain and show individual walkers versus retained ensemble steps. A trace plotted from custom flat samples only shows sample-row order. Autocorrelation also requires the actual temporal axes: interleaving walkers creates an invalid time series.

The current diagnostic averages walker autocorrelation functions. Integrated autocorrelation times are in **ensemble steps**; they can be unavailable (`NaN`) for short chains or stuck walkers. An unavailable estimate does not mean independent samples. The reliability threshold is controlled by `MCMC_AUTOCORR_TOL`; keep it unchanged to assess this run.

In [ ]:
mcmc_task.plot.traces(
    traces_to_plot=["system@eccentricity", "system@asini", "system@mass_ratio"],
)
mcmc_task.plot.autocorrelation(
    correlations_to_plot=["system@asini", "system@mass_ratio"],
)


## Reload and discard complete ensemble steps

`load_chain()` updates the current posterior summaries from the retained samples. First load a compatible **result JSON**, which supplies the full model including fixed parameters, then load the separate **chain JSON**. Loading a result alone does not restore samples; loading a chain is not a continuation of sampling.

Here an additional cut of 200 production steps is illustrative. It is not a second removal of the 200 unsaved burn-in steps, and it is not a recommendation based on convergence. Each load reads the original file: repeated identical loads do not accumulate cuts or rewrite the saved chain.

In [ ]:
retained_task = RVBinaryAnalyticsTask(
    data=rv_data,
    method="mcmc",
)
retained_task.load_result(str(raw_result_path))
additional_discard_steps = 200
retained_task.load_chain(
    filename=str(chain_path),
    discard=additional_discard_steps,
    discard_unit=DiscardUnit.STEPS,
    percentiles=[16, 50, 84],
)
expected_shape = (
    production_steps - additional_discard_steps,
    nwalkers,
    fit.chain.shape[-1],
)
np.testing.assert_equal(retained_task.fit_cls.chain.shape, expected_shape)
np.testing.assert_array_equal(
    retained_task.fit_cls.chain,
    fit.chain[additional_discard_steps:],
)
print("Retained temporal shape:", retained_task.fit_cls.chain.shape)
print("Retained rows:", retained_task.fit_cls.flat_chain.shape[0])

retained_task.load_chain(
    filename=str(chain_path),
    discard=additional_discard_steps,
    discard_unit=DiscardUnit.STEPS,
)
np.testing.assert_equal(retained_task.fit_cls.chain.shape, expected_shape)


The current discard rules are:

| File / explicit unit | Meaning of `discard` | Temporal diagnostics afterward |
|---|---|---|
| New structured export, unit omitted | Complete ensemble steps | Preserved |
| `DiscardUnit.STEPS` (or `"steps"`) | Complete ensemble steps; metadata required | Preserved |
| Legacy flat export, unit omitted | Individual flat rows | Unavailable |
| `DiscardUnit.ROWS` (or `"rows"`) | Individual flat rows | Preserved only if a structured chain is cut at a complete ensemble boundary |

With 24 walkers, 200 steps equal 4,800 rows. Removing one row cuts into an ensemble and discards temporal structure. No walker count or ordering is guessed for old flat files; requesting step discard on such a file raises an error. `discard` must be a nonnegative integer and must leave samples.

In [ ]:
# Demonstrate equivalent units on a separate task.
row_cut_task = RVBinaryAnalyticsTask(
    data=rv_data,
    method="mcmc",
)
row_cut_task.load_result(str(raw_result_path))
row_cut_task.load_chain(
    filename=str(chain_path),
    discard=additional_discard_steps * nwalkers,
    discard_unit=DiscardUnit.ROWS,
)
np.testing.assert_array_equal(
    row_cut_task.fit_cls.chain,
    retained_task.fit_cls.chain,
)

# A partial ensemble cut still leaves marginal samples, but no temporal chain.
row_cut_task.load_chain(
    filename=str(chain_path),
    discard=1,
    discard_unit=DiscardUnit.ROWS,
)
assert row_cut_task.fit_cls.chain is None
print("Partial cut: flat samples remain; ensemble time axes are unavailable.")


For an older chain containing only flat samples, use its compatible result file and explicitly request rows:

```python
legacy_task.load_result("matching_result.json")
legacy_task.load_chain(
    filename="legacy_flat_chain.json",
    discard=100,
    discard_unit=DiscardUnit.ROWS,
)
```

Corner plots and marginal percentiles remain possible, but temporal autocorrelation cannot be recovered by arbitrarily reshaping rows. Preserve original chain files and document every additional cut. There is no `thin` argument on ELISA's `load_chain()` API; thinning is not a substitute for longer sampling.

## Retained posterior summaries and plots

Default intervals use the 16th, 50th and 84th percentiles, with the median as the reported value. Only adequately sampled chains support a posterior interpretation. Corner plots show correlations and boundary effects; they do not prove convergence. `truths=True` draws the current fitted values, not known physical truth.

Parameter-value filtering with `filter_chain()` is a different operation: it conditions the posterior on selected parameter ranges, recomputes summaries and removes temporal structure. It is not burn-in removal and should not be used to hide a second mode or make uncertainties smaller.

In [ ]:
retained_task.plot.corner(
    n_bins=15,
    truths=True,
)
retained_task.plot.traces(
    traces_to_plot=["system@eccentricity", "system@asini", "system@mass_ratio"],
)
retained_task.plot.model(
    start_phase=-0.5,
    stop_phase=0.5,
    y_axis_unit=u.km / u.s,
)
retained_result_path = output_dir / "rv_mcmc_retained.json"
retained_task.save_result(str(retained_result_path))
retained_task.result_summary(
    filename=str(output_dir / "rv_mcmc_summary.txt"),
)


`result_summary(propagate_errors=True)` can propagate retained joint samples into derived quantities. This is useful once sampling is adequate; it does not repair an unconverged chain or remove the RV inclination degeneracy. Save the retained result after `load_chain()`, since its percentile summaries may differ from the all-production result.

In [ ]:
provenance = {
    "sources": ["demo_data/rv_data/rv1.dat", "demo_data/rv_data/rv2.dat"],
    "model": "kinematic double-lined RV",
    "input_velocity_unit": "km/s",
    "internal_velocity_unit": "m/s",
    "time_convention": "Bundled offset days; epoch uses the same convention",
    "nwalkers": nwalkers,
    "burn_in_steps_not_saved": burn_in_steps,
    "production_steps_saved": production_steps,
    "additional_discard": additional_discard_steps,
    "discard_unit": DiscardUnit.STEPS.value,
    "retained_chain_shape": list(retained_task.fit_cls.chain.shape),
    "percentiles": [16, 50, 84],
    "autocorrelation_tolerance": settings.MCMC_AUTOCORR_TOL,
    "status": "Short workflow demonstration; convergence is not established",
}
with (output_dir / "sampling_metadata.json").open("w") as stream:
    json.dump(provenance, stream, indent=2)
print(sorted(path.name for path in output_dir.iterdir()))


For scientific inference, extend production until temporal diagnostics are meaningful, inspect every walker, compare independent starts and check sensitivity to priors and the noise model. Weak eccentricity, angular boundaries and correlated parameters deserve particular care. Increasing walker count alone does not increase the time span explored by each walker.

The exported parameter results can inform Demo 12, while a measured inclination from photometry can help resolve the projected RV dimensions. The short MCMC intervals produced here should not be treated as final observational constraints.